<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/database.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Build an Application-Ready Marketing Transactions Schema with Sequences</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook requires a Standard or Premium workspace. Sequences are not available on Free Starter Workspaces.</p>
    </div>
</div>

## Objective

A **Real-Time Digital Marketing** conversion event becomes an application-owned **marketing transaction**. The schema keeps the source event identity for lineage and idempotency while the database generates compact, unique, sequential identifiers for application records — transactions, orders, and invoices.

This notebook shows:

* Why `MAX(id) + 1` is unsafe in concurrent or distributed systems
* How `AUTO_INCREMENT AS SEQUENCE` generates compact, unique identifiers
* How source-event identity and database-generated identity serve different purposes
* A rerun-safe insert strategy that deduplicates on the source event key

**Technical reference:** [A Developer's Guide to SingleStore Sequences](https://www.singlestore.com/blog/guide-to-singlestore-sequences/)

**Dataset:** Real-Time Digital Marketing — `purchases`, `offers`, and `vendors` tables.

**Prerequisites:** Standard or Premium workspace; no external credentials required for the core notebook.

## Section 1 — Setup

Install dependencies and establish a connection. This pattern is identical to the one used in the [Build Modern AI and Search Applications](https://www.singlestore.com/spaces/build-modern-ai-and-search-applications/) notebook.

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import random
import time
import pandas as pd
import singlestoredb as s2

pd.set_option("display.max_colwidth", 120)
random.seed(42)

DB_NAME = "mktx_sequences_demo"
USE_DB = None

def connect():
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f"USE {USE_DB}")

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    for attempt in range(4):
        try:
            return _query(query, params)
        except Exception as e:
            msg = str(e)
            if getattr(e, "errno", None) in (2006, 2013) or "Lost connection" in msg or "gone away" in msg:
                connect()
            else:
                raise

def try_sql(query, params=None, label="Step"):
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f"{label} skipped: {e}")
        return None

connect()

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == "ON"

if IS_SHARED:
    print("Shared tier detected — notebook will use the current database.")
    USE_DB = run_sql("SELECT DATABASE() AS db").iloc[0, 0]
else:
    run_sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    USE_DB = DB_NAME
    run_sql(f"USE {DB_NAME}")

connect()
print("Database:", run_sql("SELECT DATABASE() AS db").iloc[0, 0])
print("SingleStore version:", run_sql("SELECT @@memsql_version AS v").iloc[0, 0])

## Section 2 — Initialize and inspect the RTDM data

Load a bounded, deterministic sample of the RTDM `purchases`, `offers`, and `vendors` tables. These are the same tables used by the [Build Modern AI and Search Applications](https://www.singlestore.com/spaces/build-modern-ai-and-search-applications/) notebook. The `martech` database is created by the Real-Time Digital Marketing demo; if it is absent, we stream a small sample from S3 directly.

### Create the RTDM source tables

Drop and recreate the three source tables scoped to this notebook. This block is safe to rerun — `DROP TABLE IF EXISTS` ensures a clean slate.

In [3]:
# Source tables — scoped to this notebook's database
for t in ["rtdm_purchases", "rtdm_offers", "rtdm_vendors"]:
    run_sql(f"DROP TABLE IF EXISTS {t}")

run_sql('''
    CREATE TABLE rtdm_purchases (
        subscriber_id BIGINT,
        vendor_name   VARCHAR(100),
        ts            DATETIME(6) DEFAULT NOW(6),
        SORT KEY (ts),
        KEY (vendor_name) USING HASH
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE rtdm_offers (
        offer_id             BIGINT PRIMARY KEY,
        customer             VARCHAR(100),
        enabled              BOOL,
        notification_zone    TEXT,
        segment_ids          JSON,
        notification_content TEXT,
        notification_target  VARCHAR(200),
        maximum_bid_cents    BIGINT
    )
''')

run_sql('''
    CREATE ROWSTORE REFERENCE TABLE rtdm_vendors (
        domain     VARCHAR(200) PRIMARY KEY,
        vendor     VARCHAR(100),
        category   VARCHAR(50),
        popularity DOUBLE
    )
''')
print("Source tables created.")

### Load RTDM data

Use the live `martech` database if it exists, otherwise load a capped sample from S3.

**Canonical vendor enrichment — what it is and what it is not:**

The `rtdm_purchases` table stores `vendor_name` (a plain-text name decoded from base64 in the S3 source). It does not store a verified campaign domain or a confirmed purchase-to-offer relationship.

The vendors reference data contains one row per *domain* (e.g. `dabshots.mil`, `dabshots.edu`) with the same vendor *name* appearing across multiple TLDs. Joining purchases directly to vendors by name would fan out by up to 9× and inflate row counts. To prevent this, a `rtdm_vendor_canonical` view resolves each vendor name to exactly one domain using `MIN(domain)` as a deterministic tiebreaker. The `category` value comes from that same winning row.

**This is demo enrichment, not attribution:** `MIN(domain)` picks a domain that happens to sort first alphabetically — it does not represent a confirmed attribution between the subscriber's purchase event and that domain's advertising campaign. The 1:1 join only proves that canonicalization prevents row fanout; it does not prove a real campaign relationship.

In [4]:
import json
import base64, io, re, urllib.request

DEMO_DB   = "martech"
MAX_ROWS  = 10_000
BUCKET    = "https://singlestore-realtime-digital-marketing.s3.amazonaws.com"

# --- vendors ---
vdf = pd.read_json(
    "https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json"
)
vdf["domain"] = (vdf["vendor"].str.lower() + "." + vdf["tld"])
vdf = vdf.drop_duplicates("domain")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO rtdm_vendors VALUES (%s, %s, %s, %s)",
        vdf[["domain", "vendor", "category", "popularity"]].values.tolist(),
    )
print(f"Vendors loaded: {len(vdf)} rows ({vdf['vendor'].nunique()} unique names)")

# One canonical domain per vendor name.
# rtdm_purchases stores vendor_name (decoded from base64), not a verified
# source domain, so this mapping is demo enrichment — not confirmed attribution.
# MIN(domain) is the tiebreaker; category comes from that same winning row.
run_sql("""
    CREATE OR REPLACE VIEW rtdm_vendor_canonical AS
    SELECT v.vendor, v.domain, v.category
    FROM   rtdm_vendors v
    JOIN   (SELECT vendor, MIN(domain) AS domain FROM rtdm_vendors GROUP BY vendor) m
           ON m.vendor = v.vendor AND m.domain = v.domain
""")
print("Canonical vendor view created.")

# --- purchases + offers ---
demo_ok = False
try:
    n = run_sql(
        "SELECT COUNT(*) AS n FROM information_schema.TABLES "
        "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = 'purchases'", (DEMO_DB,)
    ).iloc[0, 0]
    demo_ok = int(n) > 0
except Exception:
    pass

if demo_ok:
    # martech purchases.vendor stores the plain vendor NAME.
    run_sql(f"""
        INSERT INTO rtdm_purchases (subscriber_id, vendor_name, ts)
        SELECT subscriber_id, vendor, ts
        FROM {DEMO_DB}.purchases
        ORDER BY ts DESC
        LIMIT {MAX_ROWS}
    """)
    run_sql(f"""
        INSERT IGNORE INTO rtdm_offers
        SELECT offer_id, customer, enabled, notification_zone,
               segment_ids, notification_content, notification_target, maximum_bid_cents
        FROM {DEMO_DB}.offers
        ORDER BY offer_id
        LIMIT 500
    """)
    print("Loaded from live martech database.")
else:
    # S3 NDJSON: vendor field is base64-encoded vendor name.
    print("martech DB not found — loading sample purchases from S3...")
    SCALE = "1k-2p"
    listing = urllib.request.urlopen(
        f"{BUCKET}/?list-type=2&prefix=v2/{SCALE}/purchases.", timeout=15
    ).read().decode()
    keys = re.findall(r"<Key>(.*?)</Key>", listing)[:3]
    rows_loaded = 0
    for key in keys:
        raw   = urllib.request.urlopen(f"{BUCKET}/{key}", timeout=20).read().decode()
        lines = [json.loads(l) for l in raw.strip().split("\n") if l.strip()]
        batch = []
        for row in lines:
            name = base64.b64decode(row["vendor"] + "==").decode("utf-8")
            batch.append((int(row["subscriberid"]), name))
        batch = batch[:MAX_ROWS - rows_loaded]
        with conn.cursor() as cur:
            cur.executemany(
                "INSERT INTO rtdm_purchases (subscriber_id, vendor_name) VALUES (%s, %s)",
                batch,
            )
        rows_loaded += len(batch)
        if rows_loaded >= MAX_ROWS:
            break
    # Synthesise one offer per canonical vendor domain
    run_sql("""
        INSERT INTO rtdm_offers
            (offer_id, customer, enabled, notification_zone, segment_ids,
             notification_content, notification_target, maximum_bid_cents)
        SELECT ROW_NUMBER() OVER (ORDER BY domain),
               vendor, TRUE, NULL, '[]',
               CONCAT('Deal at ', vendor), domain, 10
        FROM rtdm_vendor_canonical
    """)
    print(f"Loaded {rows_loaded} purchase rows from S3.")

run_sql('''
    SELECT
        (SELECT COUNT(*) FROM rtdm_purchases) AS purchases,
        (SELECT COUNT(*) FROM rtdm_offers)    AS offers,
        (SELECT COUNT(*) FROM rtdm_vendors)   AS vendors
''')

### Inspect a sample

Confirm the source columns that the transaction layer will consume.

In [5]:
print("--- purchases (source event: decoded vendor name) ---")
display(run_sql(
    "SELECT subscriber_id, vendor_name, ts FROM rtdm_purchases ORDER BY ts DESC LIMIT 5"
))

print("\n--- canonical vendor enrichment (one domain per vendor name, demo only) ---")
display(run_sql(
    "SELECT vendor, domain, category FROM rtdm_vendor_canonical ORDER BY vendor LIMIT 5"
))

print("\n--- row-count check: purchases after canonical enrichment join ---")
display(run_sql('''
    SELECT
        COUNT(*)                                AS purchase_rows,
        COUNT(DISTINCT p.subscriber_id)         AS unique_subscribers,
        COUNT(DISTINCT vc.domain)               AS distinct_enriched_domains,
        COUNT(DISTINCT o.offer_id)              AS distinct_offers
    FROM      rtdm_purchases       p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
'''))
# purchase_rows == unique_subscribers when each subscriber appears once.
# distinct_enriched_domains == distinct offers confirms 1:1 canonical match,
# not a verified campaign attribution.

print("\n--- offers (synthesised from canonical domains when martech DB absent) ---")
run_sql(
    "SELECT offer_id, customer, notification_target AS domain, maximum_bid_cents "
    "FROM rtdm_offers ORDER BY offer_id LIMIT 5"
)

## Section 3 — Why `MAX(id) + 1` breaks under concurrency

A common mistake is to compute the next identifier in application code:

```python
# UNSAFE — two concurrent writers can read the same MAX and produce duplicate ids
next_id = run_sql("SELECT MAX(txn_id) + 1 AS n FROM marketing_txns").iloc[0, 0]
run_sql(f"INSERT INTO marketing_txns (txn_id, ...) VALUES ({next_id}, ...)")
```

In a distributed system like SingleStore — or even in a single-node system with concurrent writes — two sessions can read the same `MAX` value in the same instant and both attempt to insert the same id, causing a duplicate-key error or silent data corruption depending on the table's constraints.

`AUTO_INCREMENT AS SEQUENCE` delegates identifier generation to the database, which coordinates allocation across aggregators without table-level locks. Each aggregator pre-caches a range of values (`sequence_min_cache_count`, default 100), so inserts are fast and unique even under high concurrency.

## Section 4 — Define the application boundary

Create three application-owned tables:

| Table | Purpose | PK type |
|---|---|---|
| `marketing_txns` | One row per conversion event accepted into the app | `INT AUTO_INCREMENT AS SEQUENCE` |
| `mktx_orders` | One order record per transaction (invoice-ready) | `INT AUTO_INCREMENT AS SEQUENCE` |
| `txn_line_items` | One row per vendor/purchase line within a transaction | `INT AUTO_INCREMENT AS SEQUENCE` |

All tables retain the original RTDM source key for lineage and rerun safety. The sequence column provides the compact application identifier. Formatted invoice references (e.g. `INV-000042`) are derived in the query layer — no stored column required.

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>One sequence per table</b></p>
        <p>SingleStore supports exactly <strong>one</strong> <code>AUTO_INCREMENT AS SEQUENCE</code> column per table. Do not add a second sequence column to the same table.</p>
    </div>
</div>

In [6]:
for t in ["txn_line_items", "mktx_orders", "marketing_txns"]:
    run_sql(f"DROP TABLE IF EXISTS {t}")

# Parent: one row per accepted conversion/purchase event
run_sql('''
    CREATE TABLE marketing_txns (
        txn_id          INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        subscriber_id   BIGINT       NOT NULL,
        event_ts        DATETIME(6)  NOT NULL,
        offer_id        BIGINT       NOT NULL,
        customer        VARCHAR(100),
        domain          VARCHAR(200),
        category        VARCHAR(50),
        bid_cents       BIGINT,
        created_at      DATETIME(6) DEFAULT NOW(6),
        UNIQUE KEY uq_src (subscriber_id, event_ts, offer_id) USING HASH
    )
''')

# Order / invoice record — one per transaction
run_sql('''
    CREATE TABLE mktx_orders (
        order_id    INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        txn_id      INT  NOT NULL,
        customer    VARCHAR(100),
        domain      VARCHAR(200),
        bid_cents   BIGINT,
        created_at  DATETIME(6) DEFAULT NOW(6),
        UNIQUE KEY uq_txn (txn_id) USING HASH
    )
''')

# Child: optional line items within a transaction
run_sql('''
    CREATE TABLE txn_line_items (
        line_id     INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        txn_id      INT  NOT NULL,
        vendor      VARCHAR(100),
        category    VARCHAR(50),
        bid_cents   BIGINT
    )
''')
print("Application tables created.")

### Verify the sequence is in place

`SHOW CREATE TABLE` confirms that `txn_id` uses the sequence mechanism.

In [7]:
run_sql("SHOW CREATE TABLE marketing_txns")

## Section 5 — Insert RTDM-derived records and validate

Insert a bounded set of enriched records from the RTDM source tables into `marketing_txns`. **Do not provide a value for `txn_id`** — the sequence generates it automatically.

Each row is joined through `rtdm_vendor_canonical` (canonical vendor enrichment) to associate a purchase event with one offer domain. This is a demo enrichment step — it does not represent a verified purchase-to-campaign attribution.

The `UNIQUE KEY uq_src (subscriber_id, event_ts, offer_id)` combined with `INSERT IGNORE` makes this cell rerun-safe: duplicate source events are silently skipped.

In [8]:
# Load 200 enriched conversion events — deterministic sample, safe to rerun.
# rtdm_vendor_canonical provides canonical vendor enrichment (one domain per
# vendor name). The resulting join is 1:1 by construction, not confirmed attribution.
run_sql('''
    INSERT IGNORE INTO marketing_txns
        (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
    SELECT
        p.subscriber_id,
        p.ts,
        o.offer_id,
        o.customer,
        vc.domain,
        COALESCE(vc.category, 'general'),
        o.maximum_bid_cents
    FROM       rtdm_purchases        p
    JOIN       rtdm_vendor_canonical vc ON vc.vendor              = p.vendor_name
    JOIN       rtdm_offers           o  ON o.notification_target  = vc.domain
    ORDER BY   p.ts DESC
    LIMIT 200
''')

# Create an order record for each transaction (one-to-one)
run_sql('''
    INSERT IGNORE INTO mktx_orders (txn_id, customer, domain, bid_cents)
    SELECT txn_id, customer, domain, bid_cents
    FROM   marketing_txns
''')

run_sql('''
    SELECT
        (SELECT COUNT(*) FROM marketing_txns) AS txns_loaded,
        (SELECT COUNT(*) FROM mktx_orders)   AS orders_created
''')

### Inspect generated transaction identifiers

The source event key (`subscriber_id`, `event_ts`, `offer_id`) sits alongside the database-generated `txn_id`. These are separate concerns:

* `txn_id` — compact application identifier, generated by the sequence, never supplied by the caller
* `(subscriber_id, event_ts, offer_id)` — source identity, used for lineage and duplicate detection

In [9]:
run_sql('''
    SELECT txn_id, subscriber_id, event_ts,
           offer_id, customer, category, bid_cents
    FROM   marketing_txns
    ORDER  BY txn_id
    LIMIT  10
''')

### Demonstrate `LAST_INSERT_ID()` for parent-child linking

When inserting into a parent-child pair, use `LAST_INSERT_ID()` to retrieve the sequence value generated for the parent row in the current session. This is the documented pattern from the [Sequences reference article](https://www.singlestore.com/blog/guide-to-singlestore-sequences/) — **do not use `MAX(txn_id)`**, which is unsafe under concurrent writes.

In [10]:
# Insert one conversion event and link its order and line items via LAST_INSERT_ID()
sample = run_sql('''
    SELECT p.subscriber_id,
           p.ts          AS event_ts,
           o.offer_id, o.customer,
           vc.domain,
           COALESCE(vc.category, 'general') AS category,
           o.maximum_bid_cents
    FROM      rtdm_purchases        p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor             = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
    ORDER BY  p.ts
    LIMIT 1
''')

row = sample.iloc[0]

with conn.cursor() as cur:
    cur.execute('''
        INSERT IGNORE INTO marketing_txns
            (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id),
           row.customer, row.domain, row.category, int(row.maximum_bid_cents)))

    # LAST_INSERT_ID() returns 0 when INSERT IGNORE skips a duplicate row.
    # Fall back to a lookup on the unique key so child rows always link correctly.
    cur.execute('SELECT LAST_INSERT_ID() INTO @last_id')
    cur.execute('''
        SELECT IF(@last_id > 0, @last_id,
            (SELECT txn_id FROM marketing_txns
             WHERE subscriber_id = %s AND event_ts = %s AND offer_id = %s))
        INTO @parent_txn_id
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id)))

    cur.execute('''
        INSERT IGNORE INTO mktx_orders (txn_id, customer, domain, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.customer, row.domain, int(row.maximum_bid_cents)))

    cur.execute('''
        INSERT INTO txn_line_items (txn_id, vendor, category, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.domain, row.category, int(row.maximum_bid_cents)))

run_sql('''
    SELECT t.txn_id,
           CONCAT('INV-', LPAD(ord.order_id, 6, '0')) AS invoice_ref,
           t.subscriber_id, t.customer, t.category,
           l.line_id, l.vendor, l.bid_cents
    FROM   marketing_txns  t
    JOIN   mktx_orders     ord ON ord.txn_id = t.txn_id
    JOIN   txn_line_items  l   ON l.txn_id   = t.txn_id
    ORDER  BY t.txn_id DESC
    LIMIT  5
''')

### Validate uniqueness

Confirm that every `txn_id` is unique and that each source event maps to exactly one transaction.

In [11]:
checks = {}

# Primary key uniqueness
dup_pk = run_sql('''
    SELECT txn_id, COUNT(*) AS cnt
    FROM   marketing_txns
    GROUP  BY txn_id
    HAVING cnt > 1
''')
checks["duplicate txn_id"] = len(dup_pk)

# Source key uniqueness
dup_src = run_sql('''
    SELECT subscriber_id, event_ts, offer_id, COUNT(*) AS cnt
    FROM   marketing_txns
    GROUP  BY subscriber_id, event_ts, offer_id
    HAVING cnt > 1
''')
checks["duplicate source key"] = len(dup_src)

# Order uniqueness (one order per transaction)
dup_ord = run_sql('''
    SELECT order_id, COUNT(*) AS cnt
    FROM   mktx_orders
    GROUP  BY order_id
    HAVING cnt > 1
''')
checks["duplicate order_id"] = len(dup_ord)

# Orphaned line items (no parent txn)
orphans = run_sql('''
    SELECT COUNT(*) AS n
    FROM   txn_line_items l
    LEFT JOIN marketing_txns t ON t.txn_id = l.txn_id
    WHERE  t.txn_id IS NULL
''')
checks["orphaned line_items"] = int(orphans.iloc[0, 0])

for k, v in checks.items():
    status = "PASS" if v == 0 else "FAIL"
    print(f"[{status}] {k}: {v}")

## Section 6 — Sequence behaviour: gaps, caching, and display identifiers

Key properties from the [Sequences reference article](https://www.singlestore.com/blog/guide-to-singlestore-sequences/):

| Property | Behaviour |
|---|---|
| **Unique** | Every generated value is guaranteed unique |
| **Sequential but not gapless** | Rolled-back transactions leave unused values; small gaps are normal |
| **Cached per aggregator** | Each aggregator pre-fetches a range (default: 100 values). Jumps up to the cache size between aggregators are expected |
| **One sequence per table** | Only one `AUTO_INCREMENT AS SEQUENCE` column is supported per table |
| **Not an idempotency key** | The sequence does not deduplicate; use a source event key or natural key for that |

**Display identifiers (`TXN-#####`, `INV-######`):**

`CONCAT('INV-', LPAD(order_id, 6, '0'))` is a *presentation-layer* label — it is derived on the fly from `mktx_orders.order_id` in the SELECT. It is **not** an independently sequence-backed invoice number. It is unique for the rows in this notebook because `order_id` (the actual sequence) is unique, but:

* It does not carry accounting-controlled, legally gapless, or source-system invoice semantics
* If you need a formal invoice identifier (e.g. for billing or compliance), create a dedicated `invoices` table with its own `INT AUTO_INCREMENT AS SEQUENCE PRIMARY KEY` column
* `INV-000042` will change if the underlying `order_id` changes — treat it as a convenience label, not a durable key

In [12]:
# Show a display representation derived in SQL, not stored
run_sql('''
    SELECT txn_id,
           CONCAT("TXN-", LPAD(txn_id, 5, "0")) AS display_ref,
           customer, category, bid_cents
    FROM   marketing_txns
    ORDER  BY txn_id
    LIMIT  8
''')

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Manual override warning</b></p>
        <p>Manually supplying a value for a sequence column (e.g. <code>INSERT INTO marketing_txns (txn_id, ...) VALUES (99, ...)</code>) can desynchronise the aggregator caches. If you ever do this, run:</p>
        <pre>AGGREGATOR SYNC AUTO_INCREMENT ON marketing_txns ALL;</pre>
        <p>This notebook does <strong>not</strong> perform manual overrides in the main path.</p>
    </div>
</div>

## Section 7 — Campaign operations query

Use the generated `txn_id` to retrieve the canonical vendor enrichment context stored with each transaction — the advertiser, domain, and offer that were associated via `rtdm_vendor_canonical` at insert time.

**Reminder:** The domain and offer_id shown here reflect canonical enrichment (the `MIN(domain)` selection), not a confirmed real-time campaign attribution from the live RTDM system. In a production deployment the source system would supply a verified offer and domain identifier at event time.

In [13]:
run_sql('''
    SELECT
        t.txn_id,
        CONCAT('TXN-', LPAD(t.txn_id, 5, '0'))       AS display_ref,
        CONCAT('INV-', LPAD(ord.order_id, 6, '0'))    AS invoice_ref,
        t.subscriber_id,
        t.event_ts,
        t.offer_id,
        t.customer                                     AS advertiser,
        t.category,
        t.domain,
        t.bid_cents,
        o.notification_content                         AS offer_content,
        o.enabled                                      AS offer_active
    FROM       marketing_txns  t
    JOIN       mktx_orders     ord ON ord.txn_id  = t.txn_id
    LEFT JOIN  rtdm_offers     o   ON o.offer_id  = t.offer_id
    ORDER BY   t.txn_id DESC
    LIMIT 5
''')

### AI-ready lookup: find similar transactions by category

The query below is runnable without AI Functions — it retrieves all transaction context needed to build a prompt or embedding. If AI Functions are enabled on your workspace, you can extend the SELECT to include `DOT_PRODUCT` similarity scoring.

In [14]:
# Runnable — no AI credentials required.
# Returns full campaign + vendor context per transaction.
# Extend with cluster.EMBED_TEXT(t.category) AS embedding if AI Functions are enabled.
run_sql('''
    SELECT
        t.txn_id,
        CONCAT('INV-', LPAD(ord.order_id, 6, '0'))  AS invoice_ref,
        t.subscriber_id,
        t.customer                                    AS advertiser,
        t.category,
        t.domain,
        t.bid_cents,
        t.event_ts
    FROM       marketing_txns  t
    JOIN       mktx_orders     ord ON ord.txn_id = t.txn_id
    WHERE      t.category IS NOT NULL
    ORDER BY   t.txn_id DESC
    LIMIT 10
''')

## Section 8 — Cleanup and rerun guidance

**To rerun the notebook from scratch:** run this cleanup cell, then re-execute from Section 2. The source RTDM tables and the application tables will be recreated cleanly.

**The `martech` database is never touched by cleanup** — only objects created by this notebook are dropped.

**Rerun safety:** The insert in Section 5 uses `INSERT IGNORE` with the `UNIQUE KEY uq_src` constraint, so re-executing individual insert cells without cleanup is also safe — duplicate source events are skipped.

In [15]:
app_tables  = ["txn_line_items", "mktx_orders", "marketing_txns"]
src_tables  = ["rtdm_purchases", "rtdm_offers", "rtdm_vendors"]

if not IS_SHARED:
    run_sql(f"DROP DATABASE IF EXISTS {DB_NAME}")
    print(f"Database {DB_NAME!r} dropped.")
else:
    for t in app_tables + src_tables:
        run_sql(f"DROP TABLE IF EXISTS {t}")
        print(f"Dropped table: {t}")
print("Cleanup complete.")

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>